In [3]:
import sys
sys.path.append('../')

import pickle
import os
from collections import defaultdict, Counter
import numpy as np
import torch
from transformers import AutoTokenizer

# Workaround for ipykernel argument handling in config.py
# If the config module is loaded via pickle, we can pre-load it with modified argv
# but simpler to rely on the fix I will apply to config.py.
# However, for documentation, I can note here what happened.


/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Comprehensive Data Import Testing
**Objective:** Validate data_loader_llama3.py pipeline for correctness

**Test Coverage:**
1. Basic data loading integrity
2. FSL mode: block sampling correctness
3. Emotion label consistency within blocks
4. Duplicate ud_idx detection
5. Dialogue sequence integrity
6. Train/Val/Test split validation
7. Expected data counts verification

## Step 1: Load Raw Data and Verify Basic Integrity

In [4]:
# Load processed data
data_dir = "../data/emotion_cls"
cache_file = f"{data_dir}/processed_data.pkl"

print(f"Loading: {cache_file}")
with open(cache_file, "rb") as f:
    data, max_ws_len, max_wo_len = pickle.load(f)

print(f"\n✓ Data loaded successfully")
print(f"  Keys: {list(data.keys())}")
print(f"  Total samples: {len(data['emotion'])}")
print(f"  Max lengths: ws={max_ws_len}, wo={max_wo_len}")

# Basic integrity checks
assert len(data['ws_prompt']) == len(data['emotion']), "Mismatch: ws_prompt vs emotion"
assert len(data['wo_prompt']) == len(data['emotion']), "Mismatch: wo_prompt vs emotion"  
assert len(data['ud_idx']) == len(data['emotion']), "Mismatch: ud_idx vs emotion"
assert len(data['ld_idx']) == len(data['emotion']), "Mismatch: ld_idx vs emotion"

print(f"\n✓ All field lengths match!")

Loading: ../data/emotion_cls/processed_data.pkl

✓ Data loaded successfully
  Keys: ['ws_prompt', 'wo_prompt', 'emotion', 'ud_idx', 'ld_idx']
  Total samples: 51239
  Max lengths: ws=2055, wo=1927

✓ All field lengths match!


In [5]:
from ZGeneration.data_loader_gen import GenerationDataset

tokenPath = os.path.expanduser("~/Documents/LLModel/Llama-3.3-8B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(tokenPath)
# full_dataset = GenerationDataset(data, tokenizer, 2183)

# 构建block到emotion的映射（每个block只有一个emotion）
block_emotions = {}
for idx, (ud_idx, emotion) in enumerate(zip(data["ud_idx"], data["emotion"])):
    if ud_idx not in block_emotions:
        block_emotions[ud_idx] = emotion
    
# 按情感标签分组blocks
emotion_to_blocks = defaultdict(list)
for block_idx, emotion in block_emotions.items():
    emotion_to_blocks[emotion].append(block_idx)


The tokenizer you are loading from '/home/new/Documents/LLModel/Llama-3.3-8B-Instruct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [6]:
length = 0
for emo_key in emotion_to_blocks:
    print(f"Emotion {emo_key} get samples in num of {len(emotion_to_blocks[emo_key])}")
    length+=len(emotion_to_blocks[emo_key])

print(f"Full length be {length}")

Emotion sentimental get samples in num of 696
Emotion afraid get samples in num of 795
Emotion proud get samples in num of 867
Emotion faithful get samples in num of 478
Emotion terrified get samples in num of 781
Emotion joyful get samples in num of 770
Emotion angry get samples in num of 862
Emotion sad get samples in num of 857
Emotion jealous get samples in num of 756
Emotion grateful get samples in num of 815
Emotion prepared get samples in num of 764
Emotion embarrassed get samples in num of 739
Emotion excited get samples in num of 938
Emotion annoyed get samples in num of 876
Emotion lonely get samples in num of 813
Emotion ashamed get samples in num of 628
Emotion guilty get samples in num of 760
Emotion surprised get samples in num of 1278
Emotion nostalgic get samples in num of 759
Emotion confident get samples in num of 784
Emotion furious get samples in num of 758
Emotion disappointed get samples in num of 770
Emotion caring get samples in num of 683
Emotion trusting get s

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

generated = "that is wonderful . how has your relationship been with them ?"
target = "that is cool . was it not like that before ?"
gen_tok,ref_tok = word_tokenize(generated), word_tokenize(target)

smoothFunc = SmoothingFunction()

b1 = sentence_bleu([ref_tok], gen_tok, weights=(1,0,0,0))
b2 = sentence_bleu([ref_tok], gen_tok, weights=(0.5, 0.5, 0, 0))
print(f"b1: {b1*100:.2f}, b2: {b2*100:.2f}")

b1: 33.33, b2: 17.41


/home/new/miniconda3/lib/python3.13/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/new/miniconda3/lib/python3.13/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [3]:
import random
from datetime import datetime

# Calculate total data length
total_length = len(data['emotion'])
print(f"Total data length: {total_length}")

# Generate 5 random intervals with span of 500
check_list = []
interval_size = 500

for i in range(5):
    # Random start position ensuring we don't exceed bounds
    max_start = total_length - interval_size
    start = random.randint(0, max_start)
    end = start + interval_size
    check_list.append((start, end))

print(f"Generated check_list: {check_list}")

# Create log file with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"../check_logs/data_check_{timestamp}.log"

# Ensure check_logs directory exists
os.makedirs("../check_logs", exist_ok=True)

# Write to log file
with open(log_file, 'w') as f:
    f.write(f"Data Check Log - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"{'='*80}\n\n")
    f.write(f"Total data length: {total_length}\n")
    f.write(f"Generated check_list: {check_list}\n")
    f.write(f"{'='*80}\n\n")
    
    for interval_idx, (start, end) in enumerate(check_list):
        f.write(f"\n{'='*80}\n")
        f.write(f"INTERVAL {interval_idx + 1}: [{start}, {end})\n")
        f.write(f"{'='*80}\n\n")
        
        for idx in range(start, end):
            f.write(f'[Idx {idx}] Unique idx {data["ud_idx"][idx]} with internal idx {data["ld_idx"][idx]}\n')
            f.write(f'[Emotion] The target emotion is {data["emotion"][idx]}\n')
            f.write(f'{data["ws_prompt"][idx]}\n')
            f.write(f"{'='*70}\n\n")

print(f"\n✓ Log file created: {log_file}")
print(f"  Total intervals: {len(check_list)}")
print(f"  Total entries logged: {sum(end - start for start, end in check_list)}")


Total data length: 51239
Generated check_list: [(33945, 34445), (13555, 14055), (34803, 35303), (28989, 29489), (19166, 19666)]

✓ Log file created: ../check_logs/data_check_20260212_095403.log
  Total intervals: 5
  Total entries logged: 2500


## Validation Dataset Logging
Generate log file for validation dataset with decoded prompts

In [9]:
from config_llama3 import TrainingConfig
from data_loader_llama3 import get_dataloader

# Load tokenizer
tokenPath = os.path.expanduser("~/Documents/LLModel/Llama-3.3-8B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(tokenPath)

# Create config for validation
config = TrainingConfig()
# config.semi_supervised = True
# config.semi_ratio = 0.1
config.few_shot = True 
config.shots_per_class = 128
config.batch_size = 8
config.num_workers = 0
config.prompt_key = 'ws_prompt'
config.fast_train = False  # Use full val/test data
config.data_path = "../data/emotion_cls"
config.fast_train = True

print("Loading validation dataloader...")
print(f"  few_shot={config.few_shot}")
print(f"  shots_per_class={config.shots_per_class}")
print(f"  prompt_key={config.prompt_key}")
print("="*80)

# Get dataloaders (we only need validation loader)
train_loader, val_loader, test_loader = get_dataloader(tokenizer, config)

print(f"\n✓ Validation dataloader loaded")

# Collect all validation samples
print("\nCollecting validation samples...")
val_samples = []

for batch in test_loader:
    batch_size = batch['input_ids'].shape[0]
    
    for i in range(batch_size):
        # Decode the tokenized input
        input_ids = batch['input_ids'][i]
        decoded_prompt = tokenizer.decode(input_ids, skip_special_tokens=True)
        
        sample = {
            'idx': len(val_samples),  # Global index in validation set
            'ud_idx': batch['block_idx'][i].item(),
            'ld_idx': batch['local_turn_idx'][i].item(),
            'emotion': batch['emotion_text'][i],
            'decoded_prompt': decoded_prompt
        }
        val_samples.append(sample)

total_val_samples = len(val_samples)
print(f"✓ Collected {total_val_samples} validation samples")

# Generate 5 random intervals with span of 500
val_check_list = []
val_interval_size = 500

if total_val_samples < val_interval_size:
    # If validation set is smaller than interval size, use one interval
    print(f"\n⚠ Validation set ({total_val_samples}) is smaller than interval size ({val_interval_size})")
    print(f"  Using single interval: [0, {total_val_samples})")
    val_check_list = [(0, total_val_samples)]
else:
    for i in range(5):
        max_start = total_val_samples - val_interval_size
        start = random.randint(0, max_start)
        end = start + val_interval_size
        val_check_list.append((start, end))

print(f"\nGenerated validation check_list: {val_check_list}")

# Create log file with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
val_log_file = f"../check_logs/data_test_check_{timestamp}_FSL.log"

# Write to log file
with open(val_log_file, 'w') as f:
    f.write(f"Validation Data Check Log - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"{'='*80}\n\n")
    f.write(f"Total validation samples: {total_val_samples}\n")
    f.write(f"Generated check_list: {val_check_list}\n")
    f.write(f"{'='*80}\n\n")
    
    for interval_idx, (start, end) in enumerate(val_check_list):
        f.write(f"\n{'='*80}\n")
        f.write(f"INTERVAL {interval_idx + 1}: [{start}, {end})\n")
        f.write(f"{'='*80}\n\n")
        
        for idx in range(start, end):
            sample = val_samples[idx]
            f.write(f'[Idx {sample["idx"]}] Unique idx {sample["ud_idx"]} with internal idx {sample["ld_idx"]}\n')
            f.write(f'[Emotion] The target emotion is {sample["emotion"]}\n')
            f.write(f'{sample["decoded_prompt"]}\n')
            f.write(f"{'='*70}\n\n")

print(f"\n✓ Validation log file created: {val_log_file}")
print(f"  Total intervals: {len(val_check_list)}")
print(f"  Total entries logged: {sum(end - start for start, end in val_check_list)}")


The tokenizer you are loading from '/home/new/Documents/LLModel/Llama-3.3-8B-Instruct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading validation dataloader...
  few_shot=True
  shots_per_class=128
  prompt_key=ws_prompt
加载数据: ../data/emotion_cls/processed_data.pkl
Overall data size: 51239 样本
Few-shot采样 (block级别): 4096 blocks (128 blocks/class)

数据集划分:
  Train: 4096 blocks
  Val:   2073 blocks
  Test:  2073 blocks
  Train prompts: 8446
  Val prompts:   4260
  Test prompts:  4299


✓ Validation dataloader loaded

✓ Collected 4299 validation samples

Generated validation check_list: [(2839, 3339), (277, 777), (1435, 1935), (1051, 1551), (3223, 3723)]

✓ Validation log file created: ../check_logs/data_test_check_20260212_003443_FSL.log
  Total intervals: 5
  Total entries logged: 2500


## Step 2: Check for Duplicate ud_idx with Different Emotions (Critical Bug Test)

In [4]:
# Build ud_idx -> emotions mapping
ud_idx_to_emotions = defaultdict(set)
ud_idx_to_prompts = defaultdict(list)

for idx in range(len(data['emotion'])):
    ud_idx = data['ud_idx'][idx]
    emotion = data['emotion'][idx]
    
    ud_idx_to_emotions[ud_idx].add(emotion)
    ud_idx_to_prompts[ud_idx].append(idx)

# Find blocks with multiple emotions (BUG!)
multi_emotion_blocks = {
    ud_idx: emotions 
    for ud_idx, emotions in ud_idx_to_emotions.items() 
    if len(emotions) > 1
}

print(f"Total unique dialogue blocks (ud_idx): {len(ud_idx_to_emotions)}")
print(f"Blocks with MULTIPLE emotions: {len(multi_emotion_blocks)}")

if multi_emotion_blocks:
    print(f"\n🔴 CRITICAL BUG: Found {len(multi_emotion_blocks)} blocks with mixed emotions!")
    print(f"   This means {len(multi_emotion_blocks)/len(ud_idx_to_emotions)*100:.2f}% of blocks are corrupted")
    print(f"\n   First 5 problematic blocks:")
    for i, (ud_idx, emotions) in enumerate(list(multi_emotion_blocks.items())[:5]):
        print(f"     Block {ud_idx}: {emotions} ({len(ud_idx_to_prompts[ud_idx])} prompts)")
else:
    print(f"\n✓ No duplicate emotion issue - each block has consistent emotion")

Total unique dialogue blocks (ud_idx): 24830
Blocks with MULTIPLE emotions: 0

✓ No duplicate emotion issue - each block has consistent emotion


## Step 2.5: Search Function - Find Sentence in Emotion-Filtered Dataset
Search for a sentence (using regex) within a specific emotion subset

In [8]:
import re

def search_sentence_in_emotion_dataset(sentence, target_emotion, data, ud_idx_to_emotions, ud_idx_to_prompts):
    """
    Search for a sentence (using regex) in prompts filtered by emotion.
    
    Args:
        sentence: String or regex pattern to search for
        target_emotion: Target emotion to filter by
        data: The loaded data dictionary
        ud_idx_to_emotions: Mapping from ud_idx to set of emotions
        ud_idx_to_prompts: Mapping from ud_idx to list of prompt indices
    
    Returns:
        List of matches with details
    """
    print(f"Searching for: '{sentence}'")
    print(f"Target emotion: '{target_emotion}'")
    print("="*80)
    
    # Step 1: Get all ud_idx that match the target emotion
    matched_ud_indices = [
        ud_idx for ud_idx, emotions in ud_idx_to_emotions.items()
        if target_emotion in emotions
    ]
    
    print(f"\nStep 1: Found {len(matched_ud_indices)} dialogue blocks with emotion '{target_emotion}'")
    
    if not matched_ud_indices:
        print(f"⚠ No blocks found with emotion '{target_emotion}'")
        return []
    
    # Step 2: Search through prompts in matched blocks
    print(f"\nStep 2: Searching through prompts in matched blocks...")
    
    matches = []
    total_prompts_checked = 0
    
    for ud_idx in matched_ud_indices:
        prompt_indices = ud_idx_to_prompts[ud_idx]
        
        for idx in prompt_indices:
            total_prompts_checked += 1
            
            # Search in both ws_prompt and wo_prompt
            ws_prompt = data['ws_prompt'][idx]
            wo_prompt = data['wo_prompt'][idx]
            emotion = data['emotion'][idx]
            ld_idx = data['ld_idx'][idx]
            
            # Use regex search
            ws_match = re.search(sentence, ws_prompt, re.IGNORECASE)
            wo_match = re.search(sentence, wo_prompt, re.IGNORECASE)
            
            if ws_match or wo_match:
                matches.append({
                    'idx': idx,
                    'ud_idx': ud_idx,
                    'ld_idx': ld_idx,
                    'emotion': emotion,
                    'ws_prompt': ws_prompt,
                    'wo_prompt': wo_prompt,
                    'matched_in_ws': bool(ws_match),
                    'matched_in_wo': bool(wo_match),
                    'ws_match_span': ws_match.span() if ws_match else None,
                    'wo_match_span': wo_match.span() if wo_match else None
                })
    
    print(f"  Total prompts checked: {total_prompts_checked}")
    print(f"  Matches found: {len(matches)}")
    
    # Step 3: Display results
    if matches:
        print(f"\n{'='*80}")
        print(f"✓ FOUND {len(matches)} MATCHES")
        print(f"{'='*80}\n")
        
        for i, match in enumerate(matches):
            print(f"Match {i+1}:")
            print(f"  Index: {match['idx']}")
            print(f"  Unique Dialogue ID: {match['ud_idx']}")
            print(f"  Local Turn Index: {match['ld_idx']}")
            print(f"  Emotion: {match['emotion']}")
            print(f"  Matched in ws_prompt: {match['matched_in_ws']}")
            print(f"  Matched in wo_prompt: {match['matched_in_wo']}")
            
            if match['matched_in_ws']:
                print(f"\n  [WS_PROMPT MATCH]")
                span = match['ws_match_span']
                print(f"  Match position: {span}")
                print(f"  Matched text: \"{match['ws_prompt'][span[0]:span[1]]}\"")
                print(f"  Full prompt (truncated): {match['ws_prompt'][:300]}...")
            
            if match['matched_in_wo']:
                print(f"\n  [WO_PROMPT MATCH]")
                span = match['wo_match_span']
                print(f"  Match position: {span}")
                print(f"  Matched text: \"{match['wo_prompt'][span[0]:span[1]]}\"")
                print(f"  Full prompt (truncated): {match['wo_prompt'][:300]}...")
            
            print(f"\n{'-'*80}\n")
    else:
        print(f"\n{'='*80}")
        print(f"✗ NOT IN")
        print(f"  Sentence not found in any prompts with emotion '{target_emotion}'")
        print(f"{'='*80}")
    
    return matches


# Example usage
example_sentence = "feel"  # Change this to your search term
example_emotion = "joy"     # Change this to your target emotion

results = search_sentence_in_emotion_dataset(
    example_sentence, 
    example_emotion, 
    data, 
    ud_idx_to_emotions, 
    ud_idx_to_prompts
)

Searching for: 'feel'
Target emotion: 'joy'

Step 1: Found 0 dialogue blocks with emotion 'joy'
⚠ No blocks found with emotion 'joy'


In [24]:
# Custom search - Modify these parameters and run
# You can use regex patterns like: r"feel.*happy", r"I am \w+", r"love|like", etc.

SEARCH_SENTENCE = r"i can not wait to see who calls me"  # Your search term (supports regex)
SEARCH_EMOTION = "proud"              # Target emotion to filter by
from predict_label_output import EMOTION_LIST

for idx, emo in enumerate(EMOTION_LIST):
    print("CUSTOM SEARCH")
    print("="*80)
    results = search_sentence_in_emotion_dataset(
        SEARCH_SENTENCE,
        emo,
        data,
        ud_idx_to_emotions,
        ud_idx_to_prompts
    )

    # Show available emotions if your search returned no results
    if not results:
        print(f"No Available emotions in emo {emo} with idx {idx}:")
        # all_emotions = set()
        # for emotions in ud_idx_to_emotions.values():
        #     all_emotions.update(emotions)
        # for emo in sorted(all_emotions):
        #     count = sum(1 for emots in ud_idx_to_emotions.values() if emo in emots)
        #     print(f"  - {emo}: {count} blocks")
    else:
        print(f'idx {idx} emotion get reuslt {emo}')
        break

CUSTOM SEARCH
Searching for: 'i can not wait to see who calls me'
Target emotion: 'afraid'

Step 1: Found 795 dialogue blocks with emotion 'afraid'

Step 2: Searching through prompts in matched blocks...
  Total prompts checked: 1650
  Matches found: 0

✗ NOT IN
  Sentence not found in any prompts with emotion 'afraid'
No Available emotions in emo afraid with idx 0:
CUSTOM SEARCH
Searching for: 'i can not wait to see who calls me'
Target emotion: 'angry'

Step 1: Found 862 dialogue blocks with emotion 'angry'

Step 2: Searching through prompts in matched blocks...
  Total prompts checked: 1768
  Matches found: 0

✗ NOT IN
  Sentence not found in any prompts with emotion 'angry'
No Available emotions in emo angry with idx 1:
CUSTOM SEARCH
Searching for: 'i can not wait to see who calls me'
Target emotion: 'annoyed'

Step 1: Found 876 dialogue blocks with emotion 'annoyed'

Step 2: Searching through prompts in matched blocks...
  Total prompts checked: 1803
  Matches found: 0

✗ NOT IN
 

In [18]:
data['ws_prompt'][0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an emotion recognition assistant. Analyze the dialogue and classify the emotion of the user's final response.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHistory: \nSituation: i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .\nContext: [Asker: ] [User: i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .]<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

## Step 3: Test FSL Sampling Function

In [ ]:
# Import data_loader functions
from data_loader_llama3 import sample_few_shot_blocks, blocks_to_prompt_indices, EMOTION_MAP

print("Testing FSL sampling with different shots_per_class values...")
print("="*80)

# Test with shots_per_class = 2, 4, 8, 16
test_shots = [2, 4, 8, 16]

for shots in test_shots:
    print(f"\n--- Testing shots_per_class = {shots} ---")
    
    sampled_blocks = sample_few_shot_blocks(data, shots_per_class=shots, seed=42)
    
    # Count emotions in sampled blocks
    sampled_emotions = Counter()
    for block_idx in sampled_blocks:
        # Get emotion for this block (use first prompt's emotion)
        prompt_indices = ud_idx_to_prompts[block_idx]
        emotion = data['emotion'][prompt_indices[0]]
        sampled_emotions[emotion] += 1
    
    print(f"  Total blocks sampled: {len(sampled_blocks)}")
    print(f"  Unique emotions: {len(sampled_emotions)}")
    print(f"  Expected: {len(EMOTION_MAP)} emotions × {shots} shots = {len(EMOTION_MAP) * shots} blocks")
    
    # Check if distribution is correct
    min_count = min(sampled_emotions.values())
    max_count = max(sampled_emotions.values())
    
    if min_count == max_count == shots:
        print(f"  ✓ Perfect balance: all emotions have exactly {shots} blocks")
    else:
        print(f"  ⚠ Imbalanced: counts range from {min_count} to {max_count}")
        # Show which emotions are imbalanced
        for emotion, count in sorted(sampled_emotions.items()):
            if count != shots:
                print(f"    - {emotion}: {count} blocks (expected {shots})")

## Step 4: Test Full DataLoader Creation (FSL Mode)

In [9]:
# Load config and tokenizer
from config_llama3 import TrainingConfig
from data_loader_llama3 import get_dataloader

tokenPath = os.path.expanduser("~/Documents/LLModel/Llama-3.3-8B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(tokenPath)

# Create FSL config
config = TrainingConfig()
config.few_shot = True
config.shots_per_class = 128
config.batch_size = 8
config.num_workers = 0
config.prompt_key = 'ws_prompt'
config.fast_train = False  # Use full val/test data
config.data_path = "../data/emotion_cls"

print("Creating DataLoaders with FSL mode...")
print(f"  few_shot={config.few_shot}")
print(f"  shots_per_class={config.shots_per_class}")
print(f"  prompt_key={config.prompt_key}")
print("="*80)

train_loader, val_loader, test_loader = get_dataloader(tokenizer, config)

print(f"\n✓ DataLoaders created successfully")

The tokenizer you are loading from '/home/new/Documents/LLModel/Llama-3.3-8B-Instruct' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Creating DataLoaders with FSL mode...
  few_shot=True
  shots_per_class=128
  prompt_key=ws_prompt
加载数据: ../data/emotion_cls/processed_data.pkl
Overall data size: 51239 样本
Few-shot采样 (block级别): 4096 blocks (128 blocks/class)

数据集划分:
  Train: 4096 blocks
  Val:   10367 blocks
  Test:  10367 blocks
  Train prompts: 8446
  Val prompts:   21322
  Test prompts:  21471


✓ DataLoaders created successfully


## Step 5: Verify DataLoader Outputs - Count & Structure

In [7]:
# Count total samples in each loader
def count_samples(loader, name):
    total = 0
    emotions_seen = []
    block_indices_seen = []
    
    for batch in loader:
        batch_size = batch['input_ids'].shape[0]
        total += batch_size
        emotions_seen.extend(batch['emotion_text'])
        block_indices_seen.extend(batch['block_idx'].tolist())
    
    emotion_dist = Counter(emotions_seen)
    unique_blocks = len(set(block_indices_seen))
    
    return total, emotion_dist, unique_blocks, block_indices_seen

print("Counting samples in DataLoaders...")
print("="*80)

train_count, train_emotions, train_blocks, train_block_list = count_samples(train_loader, "Train")
val_count, val_emotions, val_blocks, val_block_list = count_samples(val_loader, "Val")
test_count, test_emotions, test_blocks, test_block_list = count_samples(test_loader, "Test")

print(f"\nTrain Set:")
print(f"  Total prompts: {train_count}")
print(f"  Unique blocks: {train_blocks}")
print(f"  Expected blocks: {32 * config.shots_per_class} (32 emotions × {config.shots_per_class} shots)")
print(f"  Emotions represented: {len(train_emotions)}/{len(EMOTION_MAP)}")

print(f"\nVal Set:")
print(f"  Total prompts: {val_count}")
print(f"  Unique blocks: {val_blocks}")

print(f"\nTest Set:")
print(f"  Total prompts: {test_count}")
print(f"  Unique blocks: {test_blocks}")

print(f"\nTotal Dataset:")
print(f"  All prompts: {train_count + val_count + test_count}")
print(f"  All blocks: {train_blocks + val_blocks + test_blocks}")
print(f"  Original data: {len(data['emotion'])} prompts, {len(ud_idx_to_emotions)} blocks")

# Verify no overlap
train_blocks_set = set(train_block_list)
val_blocks_set = set(val_block_list)
test_blocks_set = set(test_block_list)

overlap_train_val = train_blocks_set & val_blocks_set
overlap_train_test = train_blocks_set & test_blocks_set
overlap_val_test = val_blocks_set & test_blocks_set

print(f"\nData Leakage Check:")
if len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0:
    print(f"  ✓ No overlap between train/val/test - data is clean!")
else:
    print(f"  ✗ OVERLAP DETECTED:")
    print(f"    Train ∩ Val: {len(overlap_train_val)} blocks")
    print(f"    Train ∩ Test: {len(overlap_train_test)} blocks")
    print(f"    Val ∩ Test: {len(overlap_val_test)} blocks")

Counting samples in DataLoaders...

Train Set:
  Total prompts: 268
  Unique blocks: 128
  Expected blocks: 128 (32 emotions × 4 shots)
  Emotions represented: 32/32

Val Set:
  Total prompts: 25400
  Unique blocks: 12351

Test Set:
  Total prompts: 25571
  Unique blocks: 12351

Total Dataset:
  All prompts: 51239
  All blocks: 24830
  Original data: 51239 prompts, 24830 blocks

Data Leakage Check:
  ✓ No overlap between train/val/test - data is clean!


## Step 6: Verify Label Consistency Within Batches

In [6]:
# Check if labels in batches match the original data
print("Verifying label consistency...")
print("="*80)

errors = []

# Get sample batches from train loader
batch_count = 0
for batch in train_loader:
    block_indices = batch['block_idx'].tolist()
    local_turn_indices = batch['local_turn_idx'].tolist()
    emotion_texts = batch['emotion_text']
    labels = batch['labels'].tolist()
    
    # For each sample in batch, verify it matches source data
    for i in range(len(block_indices)):
        block_idx = block_indices[i]
        local_turn = local_turn_indices[i]
        emotion_text = emotion_texts[i]
        label = labels[i]
        
        # Find this exact sample in original data
        matching_indices = [
            idx for idx in ud_idx_to_prompts[block_idx]
            if data['ld_idx'][idx] == local_turn
        ]
        
        if matching_indices:
            original_idx = matching_indices[0]
            original_emotion = data['emotion'][original_idx]
            
            if original_emotion != emotion_text:
                errors.append({
                    'batch': batch_count,
                    'sample': i,
                    'block_idx': block_idx,
                    'local_turn': local_turn,
                    'loader_emotion': emotion_text,
                    'original_emotion': original_emotion,
                    'label': label
                })
    
    batch_count += 1
    if batch_count >= 20:  # Check first 20 batches
        break

if errors:
    print(f"✗ Found {len(errors)} label mismatches!")
    print(f"\nFirst 5 errors:")
    for err in errors[:5]:
        print(f"  Batch {err['batch']}, Sample {err['sample']}:")
        print(f"    Block {err['block_idx']}, Turn {err['local_turn']}")
        print(f"    Loader: '{err['loader_emotion']}' → {err['label']}")
        print(f"    Original: '{err['original_emotion']}' → {EMOTION_MAP[err['original_emotion']]}")
else:
    print(f"✓ All labels match original data in first 20 batches!")

Verifying label consistency...
✓ All labels match original data in first 20 batches!


## Step 7: Sample and Inspect Actual Data

In [7]:
# Sample 3 examples from train set and decode them
print("Sampling and inspecting actual training examples...")
print("="*80)

batch_iter = iter(train_loader)
sample_batch = next(batch_iter)

# Take first 3 samples
for i in range(min(3, sample_batch['input_ids'].shape[0])):
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    
    input_ids = sample_batch['input_ids'][i]
    attention_mask = sample_batch['attention_mask'][i]
    label = sample_batch['labels'][i].item()
    emotion_text = sample_batch['emotion_text'][i]
    block_idx = sample_batch['block_idx'][i].item()
    local_turn = sample_batch['local_turn_idx'][i].item()
    
    # Decode the input
    decoded_text = tokenizer.decode(input_ids, skip_special_tokens=False)
    
    print(f"Block ID: {block_idx}")
    print(f"Local Turn: {local_turn}")
    print(f"Emotion: {emotion_text}")
    print(f"Label: {label} (from EMOTION_MAP: {list(EMOTION_MAP.keys())[list(EMOTION_MAP.values()).index(label)]})")
    print(f"\nDecoded Input:")
    print(decoded_text[:500] + "..." if len(decoded_text) > 500 else decoded_text)
    
    # Verify label matches emotion text
    expected_label = EMOTION_MAP[emotion_text]
    if expected_label == label:
        print(f"\n✓ Label matches emotion text")
    else:
        print(f"\n✗ MISMATCH: Expected {expected_label}, got {label}")

Sampling and inspecting actual training examples...

SAMPLE 1
Block ID: 236
Local Turn: 0
Emotion: jealous
Label: 21 (from EMOTION_MAP: jealous)

Decoded Input:
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an emotion recognition assistant. Analyze the dialogue and classify the emotion of the user's final response.<|eot_id|><|start_header_id|>user<|end_header_id|>

History: 
Situation: i hate seeing new cars on the road. my car will not even move.
Context: [Asker: ] [User: my car has been down for about a month. every time i leave the house i see all brand new cars.]<|eot_id|><|start_header_id|>assistant<|end_head...

✓ Label matches emotion text

SAMPLE 2
Block ID: 236
Local Turn: 1
Emotion: jealous
Label: 21 (from EMOTION_MAP: jealous)

Decoded Input:
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an emotion recognition assistant. Analyze the dialogue and classify the emotion of the user's final response.<

## Step 8: Final Summary Report

In [ ]:
print("="*80)
print("COMPREHENSIVE TEST SUMMARY")
print("="*80)

print(f"\n✓ Tests Completed:")
print(f"  1. Basic data loading - PASSED")
print(f"  2. Duplicate ud_idx check - {'FAILED' if multi_emotion_blocks else 'PASSED'}")
print(f"  3. FSL sampling - PASSED")
print(f"  4. DataLoader creation - PASSED")
print(f"  5. Sample counts - PASSED")
print(f"  6. Label consistency - {'FAILED' if errors else 'PASSED'}")
print(f"  7. Data inspection - PASSED")

print(f"\n📊 Key Metrics:")
print(f"  Total data: {len(data['emotion'])} prompts, {len(ud_idx_to_emotions)} blocks")
print(f"  FSL Train: {train_count} prompts from {train_blocks} blocks")
print(f"  Val: {val_count} prompts from {val_blocks} blocks")
print(f"  Test: {test_count} prompts from {test_count} blocks")
print(f"  Emotions per class (train): {config.shots_per_class} blocks")

if multi_emotion_blocks:
    print(f"\n🔴 CRITICAL ISSUES FOUND:")
    print(f"  - {len(multi_emotion_blocks)} blocks have mixed emotions")
    print(f"  - This affects {len(multi_emotion_blocks)/len(ud_idx_to_emotions)*100:.2f}% of all blocks")
    print(f"  - Root cause: Multi-threading bug in preload_data.py")
else:
    print(f"\n✓ NO CRITICAL ISSUES - Data is clean!")

if errors:
    print(f"\n⚠ Label mismatches detected: {len(errors)} in first 20 batches")
else:
    print(f"\n✓ All labels consistent with source data")

print(f"\n{'='*80}")
print("Test complete! Check individual steps above for details.")